In [ ]:
# ── 최종 요약 출력 ──
print("=" * 70)
print("  Classical DSP 축 불균형 진단 — 최종 요약")
print("=" * 70)
print()
print(f"[시스템 사양]")
print(f"  파쇄기 A축: {RPM_A} RPM ({F1X_A:.1f} Hz)")
print(f"  파쇄기 B축: {RPM_B} RPM ({F1X_B:.2f} Hz)")
print(f"  샘플링 주파수: {FS:,} Hz")
print(f"  FFT 분해능: {1/T_WINDOW:.1f} Hz (윈도우: {T_WINDOW}s)")
print()
print(f"[데이터 규모]")
print(f"  시나리오별 FFT: {N_FFT:,} samples × {len(SCENARIOS)} = {N_FFT * len(SCENARIOS):,}")
print(f"  90일 트렌드: {TREND_SAMPLES:,} samples ({TREND_DAYS}일 × 60초 간격)")
print(f"  스펙트럼 워터폴: {n_snapshots} snapshots × {len(freqs_display):,} freq bins")
print()
print(f"[열화 분석 결과]")
print(f"  선형 열화율: {slope:.4f} mm/s/day")
print(f"  지수 열화율: {exp_rate:.5f} /day")
print()
print(f"[임계값 도달 예측 (지수 모델)]")
for label, threshold in thresholds.items():
    if exp_rate > 0 and exp_intercept > 0:
        day_pred = np.log(threshold / exp_intercept) / exp_rate
        if day_pred > TREND_DAYS:
            print(f"  {label}: Day {day_pred:.0f} (잔여 {day_pred - TREND_DAYS:.0f}일)")
        else:
            print(f"  {label}: Day {day_pred:.0f} (이미 초과)")
print()
print("=" * 70)
print("  분석 완료 — 본 결과를 기반으로 정비 계획을 수립하십시오.")
print("=" * 70)

---
## Step 7. 종합 요약 및 실무 적용

### ISO 10816-3 진동 심각도 기준표

| Zone | 진동 속도 (mm/s RMS) | 상태 | 권장 조치 | 점검 주기 |
|------|---------------------|------|----------|----------|
| **A** | < 2.5 | Good (양호) | 정상 운전 | 월 1회 |
| **B** | 2.5 ~ 6.3 | Acceptable (허용) | 모니터링 강화 | 주 1회 |
| **C** | 6.3 ~ 10.0 | Warning (경고) | 정비 계획 수립 | 일 1회 |
| **D** | > 10.0 | Danger (위험) | **즉시 정지 및 정비** | 실시간 |

### 축 불균형 진단의 핵심 포인트

1. **1X 주파수 성분이 핵심**: 축 불균형은 회전 주파수(1X)에서 가장 큰 에너지를 발생시킵니다.
2. **2X/3X 비율 확인**: 2X가 1X보다 큰 경우 → 미스얼라인먼트 의심, 3X 증가 → 구조 결함 의심
3. **트렌드 모니터링 필수**: 단일 측정값보다 **시간에 따른 변화 추세**가 더 중요합니다.
4. **지수적 열화 패턴**: 기계 결함은 선형이 아닌 **지수적으로 가속**되므로, 초기 대응이 중요합니다.

### 실무 배포 시 고려사항

| 항목 | 권장 사항 |
|------|----------|
| 센서 | ICP 가속도계 (100mV/g), 베어링 하우징 수직/수평 설치 |
| 샘플링 | 10 kHz 이상 (1X 주파수의 최소 10배) |
| FFT 윈도우 | 1초 (주파수 분해능 1 Hz) |
| 트렌드 저장 | 60초 간격 1X 진폭 RMS 값 |
| 알람 설정 | Zone C 진입 시 경고, Zone D 진입 시 비상 정지 |
| 데이터 보관 | 최소 90일 (열화 추세 분석용) |
| 밸런싱 기준 | ISO 1940 G6.3 (파쇄기 등급) |

### 분석 파이프라인 요약

```
Raw Vibration Signal (10kHz)
    ↓ Hanning Window
FFT Spectrum
    ↓ Peak Detection
1X / 2X / 3X Amplitudes
    ↓ ISO 10816-3 Classification
Zone A/B/C/D → Alert/Action
    ↓ Trend Storage (60s interval)
90-Day Historical Data
    ↓ Regression Analysis
Remaining Useful Life Prediction
```

In [ ]:
# ── 6-6. Predicted Time to Threshold Crossing ──
fig, ax = plt.subplots(figsize=(16, 8))

y_max_pred = 18

# ISO Zone 배경
ax.axhspan(0, ISO_ZONE_A, alpha=0.10, color=ISO_COLORS['A'])
ax.axhspan(ISO_ZONE_A, ISO_ZONE_B, alpha=0.10, color=ISO_COLORS['B'])
ax.axhspan(ISO_ZONE_B, ISO_ZONE_C, alpha=0.10, color=ISO_COLORS['C'])
ax.axhspan(ISO_ZONE_C, y_max_pred, alpha=0.10, color=ISO_COLORS['D'])

# ISO 경계선
for th, lbl, col in [(ISO_ZONE_A, 'Zone A/B: 2.5 mm/s', ISO_COLORS['B']),
                      (ISO_ZONE_B, 'Zone B/C: 6.3 mm/s', ISO_COLORS['C']),
                      (ISO_ZONE_C, 'Zone C/D: 10.0 mm/s', ISO_COLORS['D'])]:
    ax.axhline(y=th, color=col, linestyle='--', linewidth=1.5, alpha=0.7)
    ax.text(122, th + 0.2, lbl, fontsize=8, color=col, fontweight='bold')

# 24시간 이동 평균 (실측)
ax.plot(trend_days, trend_ma_long, color='#2c3e50', linewidth=2.0,
        label='24h Moving Average (Observed)', zorder=3)

# 현재 시점 표시
ax.axvline(x=TREND_DAYS, color='black', linestyle='-', linewidth=1.5, alpha=0.5)
ax.text(TREND_DAYS + 0.5, y_max_pred * 0.95, 'Current\n(Day 90)',
        fontsize=9, fontweight='bold', va='top')

# 예측 구간 (회색 배경)
ax.axvspan(TREND_DAYS, TREND_DAYS + 30, alpha=0.06, color='gray')
ax.text(TREND_DAYS + 15, y_max_pred * 0.98, 'Prediction Window',
        fontsize=9, ha='center', va='top', style='italic', color='gray')

# 선형 회귀 예측선
ax.plot(future_days, pred_linear, color='#3498db', linewidth=2.0,
        linestyle='--', alpha=0.8, label=f'Linear Regression (slope={slope:.3f}/day)')

# 지수 회귀 예측선
pred_exp_clipped = np.minimum(pred_exp, y_max_pred * 1.5)
ax.plot(future_days, pred_exp_clipped, color='#e74c3c', linewidth=2.0,
        linestyle='-.', alpha=0.8, label=f'Exponential Fit (rate={exp_rate:.4f}/day)')

# 임계값 교차점 마커
for label, threshold in [('Zone B', ISO_ZONE_A), ('Zone C', ISO_ZONE_B), ('Zone D', ISO_ZONE_C)]:
    # 선형 모델
    day_lin = (threshold - intercept) / slope if slope > 0 else None
    # 지수 모델
    day_exp = np.log(threshold / exp_intercept) / exp_rate if exp_rate > 0 and exp_intercept > 0 else None

    if day_exp is not None and 0 < day_exp <= TREND_DAYS + 30:
        ax.plot(day_exp, threshold, 'v', markersize=12, color='#e74c3c',
                zorder=5, markeredgecolor='white', markeredgewidth=1.5)
        ax.annotate(f'{label}\nDay {day_exp:.0f}',
                    xy=(day_exp, threshold),
                    xytext=(day_exp + 3, threshold + 1.2),
                    fontsize=8, fontweight='bold', color='#e74c3c',
                    arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.2))

ax.set_xlim(0, TREND_DAYS + 30)
ax.set_ylim(0, y_max_pred)
ax.set_xlabel('Time (Days)', fontsize=12)
ax.set_ylabel('1X Vibration Amplitude (mm/s)', fontsize=12)
ax.set_title('Remaining Useful Life Prediction — Threshold Crossing Estimation',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()
print("그림 6-6: 잔여 수명 예측 — 지수 모델이 실제 열화 패턴에 더 부합합니다.")

### 6-6. 잔여 수명 예측 (Remaining Useful Life Prediction)

선형 및 지수 회귀 모델을 통해 **ISO Zone D 임계값(10 mm/s) 도달 시점**을 예측합니다.  
이를 통해 **사전 정비 계획(Predictive Maintenance)** 수립이 가능합니다.

In [ ]:
# ── 6-5. Spectrum Waterfall (Heatmap over Time) ──

# 90일간 매일 1회 FFT 스냅샷 생성 (90개 시점)
n_snapshots = 90
snapshot_days = np.linspace(0, TREND_DAYS - 1, n_snapshots)
freq_display_max = 100  # Hz

waterfall_data = []
np.random.seed(123)

for day in snapshot_days:
    # 해당 시점의 1X 진폭 (트렌드에서 추출)
    idx = int(day * 24 * 60)
    idx = min(idx, len(trend_1x) - 1)
    current_1x = trend_ma_long[idx]  # 이동 평균 값 사용

    # 해당 시점의 진동 신호 합성
    params = {
        '1X': current_1x,
        '2X_ratio': 0.35 + 0.002 * day,  # 2X 비율도 서서히 증가
        '3X_ratio': 0.12 + 0.001 * day,
        'noise': 0.3 + 0.01 * day,
    }
    _, sig = generate_vibration_signal(params, f1x=F1X_A, fs=FS, duration=0.5)
    freqs_wf, mag_wf = compute_fft(sig, fs=FS)

    # 주파수 범위 제한
    freq_mask = freqs_wf <= freq_display_max
    waterfall_data.append(mag_wf[freq_mask])

freqs_display = freqs_wf[freq_mask]
waterfall_matrix = np.array(waterfall_data)

# 히트맵 시각화
fig, ax = plt.subplots(figsize=(14, 8))

im = ax.pcolormesh(freqs_display, snapshot_days, waterfall_matrix,
                    cmap='hot', shading='gouraud', vmin=0,
                    vmax=np.percentile(waterfall_matrix, 98))

cbar = plt.colorbar(im, ax=ax, label='Amplitude (mm/s)', shrink=0.9)

# 1X, 2X, 3X 위치 표시
for h, label in [(F1X_A, '1X'), (2*F1X_A, '2X'), (3*F1X_A, '3X')]:
    ax.axvline(x=h, color='cyan', linestyle='--', linewidth=1.2, alpha=0.7)
    ax.text(h + 0.5, snapshot_days[-1] + 1, label, color='cyan',
            fontsize=10, fontweight='bold', va='bottom')

ax.set_xlabel('Frequency (Hz)', fontsize=12)
ax.set_ylabel('Time (Days)', fontsize=12)
ax.set_title('Spectrum Waterfall — 90-Day Frequency Evolution (Heatmap)',
             fontsize=14, fontweight='bold')
ax.set_xlim(0, freq_display_max)

plt.tight_layout()
plt.show()
print("그림 6-5: 스펙트럼 워터폴 — 1X 대역의 에너지가 시간에 따라 증가하는 열화 패턴이 명확합니다.")

### 6-5. 스펙트럼 워터폴 (Spectrum Waterfall — Heatmap)

90일간의 주파수 스펙트럼 변화를 히트맵으로 시각화합니다.  
시간이 경과함에 따라 **1X 주파수 대역의 에너지가 증가**하는 패턴을 관찰할 수 있습니다.

In [ ]:
# ── 6-4. ISO Classification Pie Chart ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 왼쪽: 파이 차트
zone_counts = {}
for z in ['A', 'B', 'C', 'D']:
    zone_counts[z] = np.sum(zones == z)

labels_pie = [f'Zone {z}: {ISO_LABELS[z]}\n({count:,} pts, {100*count/len(zones):.1f}%)'
              for z, count in zone_counts.items() if count > 0]
sizes = [count for count in zone_counts.values() if count > 0]
colors_pie = [ISO_COLORS[z] for z, count in zone_counts.items() if count > 0]
explode = [0.05] * len(sizes)

wedges, texts, autotexts = ax1.pie(
    sizes, explode=explode, labels=None, autopct='%1.1f%%',
    colors=colors_pie, startangle=90, pctdistance=0.75,
    wedgeprops=dict(linewidth=2, edgecolor='white')
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')

ax1.legend(labels_pie, loc='center left', bbox_to_anchor=(-0.35, 0.5), fontsize=9)
ax1.set_title('ISO 10816-3 Zone Distribution\n(90-Day Trend Data)', fontsize=13, fontweight='bold')

# 오른쪽: 막대 그래프 (시간대별 Zone 비율)
n_weeks = 13  # ~90일 = 13주
week_samples = len(trend_1x) // n_weeks

zone_weekly = {'A': [], 'B': [], 'C': [], 'D': []}
week_labels = []

for w in range(n_weeks):
    start = w * week_samples
    end = min((w + 1) * week_samples, len(trend_1x))
    week_zones = zones[start:end]
    total = len(week_zones)
    for z in ['A', 'B', 'C', 'D']:
        zone_weekly[z].append(100 * np.sum(week_zones == z) / total)
    week_labels.append(f'W{w+1}')

x_weeks = np.arange(n_weeks)
bottom = np.zeros(n_weeks)

for z in ['A', 'B', 'C', 'D']:
    values = np.array(zone_weekly[z])
    ax2.bar(x_weeks, values, bottom=bottom, color=ISO_COLORS[z],
            label=f'Zone {z}: {ISO_LABELS[z]}', edgecolor='white', linewidth=0.5)
    bottom += values

ax2.set_xticks(x_weeks)
ax2.set_xticklabels(week_labels, fontsize=8)
ax2.set_xlabel('Week', fontsize=11)
ax2.set_ylabel('Percentage (%)', fontsize=11)
ax2.set_title('Weekly ISO Zone Distribution', fontsize=13, fontweight='bold')
ax2.legend(loc='upper left', fontsize=8)
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()
print("그림 6-4: ISO Zone 분포 — 시간이 경과할수록 Zone C/D 비율이 증가하는 열화 패턴이 확인됩니다.")

### 6-4. ISO 분류 분포 (파이 차트)

90일간 데이터의 ISO Zone별 분포를 파이 차트로 표시합니다.  
정비 시점 판단의 근거 자료로 활용됩니다.

In [ ]:
# ── 6-3. 90-Day 1X Amplitude Trend with ISO Zones ──
fig, ax = plt.subplots(figsize=(16, 7))

y_max = max(trend_1x.max(), 15) * 1.1

# ISO Zone 배경 (수평 대역)
ax.axhspan(0, ISO_ZONE_A, alpha=0.12, color=ISO_COLORS['A'], label='Zone A: Good (<2.5)')
ax.axhspan(ISO_ZONE_A, ISO_ZONE_B, alpha=0.12, color=ISO_COLORS['B'], label='Zone B: Acceptable (2.5-6.3)')
ax.axhspan(ISO_ZONE_B, ISO_ZONE_C, alpha=0.12, color=ISO_COLORS['C'], label='Zone C: Warning (6.3-10)')
ax.axhspan(ISO_ZONE_C, y_max, alpha=0.12, color=ISO_COLORS['D'], label='Zone D: Danger (>10)')

# ISO 경계선
for threshold, lbl in [(ISO_ZONE_A, '2.5'), (ISO_ZONE_B, '6.3'), (ISO_ZONE_C, '10.0')]:
    ax.axhline(y=threshold, color='gray', linestyle='--', linewidth=1, alpha=0.7)
    ax.text(91, threshold + 0.15, f'{lbl} mm/s', fontsize=8, color='gray', va='bottom')

# 원시 데이터 (얇게)
ax.plot(trend_days, trend_1x, color='#3498db', linewidth=0.2, alpha=0.4, label='Raw data (60s interval)')

# 4시간 이동 평균
ax.plot(trend_days, trend_ma, color='#2980b9', linewidth=1.0, alpha=0.8, label='4-hour Moving Average')

# 24시간 이동 평균
ax.plot(trend_days, trend_ma_long, color='#e74c3c', linewidth=2.0, alpha=0.9, label='24-hour Moving Average')

ax.set_xlim(0, 90)
ax.set_ylim(0, y_max)
ax.set_xlabel('Time (Days)', fontsize=12)
ax.set_ylabel('1X Vibration Amplitude (mm/s)', fontsize=12)
ax.set_title('90-Day Shaft Imbalance Trend — 1X Vibration Amplitude with ISO 10816-3 Zones',
             fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)

# 데이터 포인트 수 표시
ax.text(0.98, 0.02, f'Total: {len(trend_1x):,} data points',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.show()
print("그림 6-3: 90일 트렌드 — 지수적 열화 패턴과 돌발 이벤트가 관찰됩니다.")

### 6-3. 90일 1X 진폭 트렌드 (ISO Zone 배경)

129,600개 데이터포인트의 90일 트렌드를 ISO 10816-3 Zone 배경과 함께 표시합니다.  
이동 평균선으로 장기 열화 추세를 확인할 수 있습니다.

In [ ]:
# ── 6-2. FFT Frequency Spectrum with Harmonic Peaks ──
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)

freq_max = 100  # 0~100 Hz 범위 표시

for i, (name, data) in enumerate(fft_results.items()):
    ax = axes[i]
    freqs = data['freqs']
    mag = data['magnitude']
    harmonics = data['harmonics']
    color = SCENARIOS[name]['color']

    # 주파수 범위 필터
    mask = freqs <= freq_max
    ax.plot(freqs[mask], mag[mask], color=color, linewidth=0.8, alpha=0.9)
    ax.fill_between(freqs[mask], mag[mask], alpha=0.15, color=color)

    # 1X, 2X, 3X 피크 마커
    markers = {'1X': 'o', '2X': 's', '3X': 'D'}
    for h_label, (h_freq, h_amp) in harmonics.items():
        ax.plot(h_freq, h_amp, marker=markers[h_label], markersize=8,
                color='red', zorder=5)
        ax.annotate(f'{h_label}\n{h_amp:.1f} mm/s',
                    xy=(h_freq, h_amp),
                    xytext=(h_freq + 5, h_amp + 0.3),
                    fontsize=8, fontweight='bold', color='red',
                    arrowprops=dict(arrowstyle='->', color='red', lw=1))

    ax.set_ylabel(f'{name}\n(mm/s)', fontsize=10, fontweight='bold')
    ax.set_xlim(0, freq_max)

axes[0].set_title('FFT Frequency Spectrum — Harmonic Analysis (1X / 2X / 3X)',
                   fontsize=14, fontweight='bold', pad=10)
axes[-1].set_xlabel('Frequency (Hz)', fontsize=12)

# 1X, 2X, 3X 주파수 위치 표시 (세로 점선)
for ax in axes:
    for h, ls in [(F1X_A, '--'), (2*F1X_A, ':'), (3*F1X_A, '-.')]:
        ax.axvline(x=h, color='gray', linestyle=ls, alpha=0.4, linewidth=0.8)

plt.tight_layout()
plt.show()
print("그림 6-2: FFT 스펙트럼 — 1X 피크가 불균형의 핵심 지표이며, 심각도에 비례하여 증가합니다.")

### 6-2. FFT 주파수 스펙트럼 (1X / 2X / 3X 피크 표시)

각 시나리오의 주파수 스펙트럼에서 **1X, 2X, 3X 고조파 피크**를 표시합니다.  
축 불균형의 핵심 지표인 **1X 성분이 지배적**인 것이 특징입니다.

In [ ]:
# ── 6-1. Time-Domain Vibration Waveform (50ms Zoom) ──
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)

zoom_ms = 50  # 50ms 확대
zoom_samples = int(FS * zoom_ms / 1000)

for i, (name, data) in enumerate(signals.items()):
    ax = axes[i]
    t_ms = data['t'][:zoom_samples] * 1000  # ms 변환
    sig = data['signal'][:zoom_samples]
    color = data['params']['color']

    ax.plot(t_ms, sig, color=color, linewidth=0.8, alpha=0.9)
    ax.fill_between(t_ms, sig, alpha=0.15, color=color)
    ax.set_ylabel(f'{name}\n(mm/s)', fontsize=10, fontweight='bold')
    ax.set_ylim(-22, 22)

    # RMS 표시
    rms = np.sqrt(np.mean(data['signal']**2))
    ax.text(0.98, 0.92, f'RMS = {rms:.2f} mm/s',
            transform=ax.transAxes, ha='right', va='top',
            fontsize=9, bbox=dict(boxstyle='round,pad=0.3',
                                   facecolor=color, alpha=0.3))

axes[0].set_title('Time-Domain Vibration Waveform — 5 Imbalance Scenarios (50ms Zoom)',
                   fontsize=14, fontweight='bold', pad=10)
axes[-1].set_xlabel('Time (ms)', fontsize=12)

plt.tight_layout()
plt.savefig('/content/fig_6_1_time_domain.png', dpi=150, bbox_inches='tight') if False else None
plt.show()
print("그림 6-1: 시간 영역 진동 파형 — 불균형 심각도에 따라 진폭이 증가하는 것을 확인할 수 있습니다.")

### 6-1. 시간 영역 진동 파형 (50ms 확대)

5가지 불균형 시나리오의 시간 영역 신호를 50ms 구간으로 확대하여 비교합니다.  
불균형이 심할수록 **진폭이 크고 1X 패턴이 뚜렷**해집니다.

---
## Step 6. 시각화 (Visualization)

각 차트를 개별 셀에서 렌더링합니다.  
모든 matplotlib 텍스트(제목, 축 레이블, 범례)는 **영어**로 표기합니다.

In [ ]:
# ── 이동 평균 계산 ──
def moving_average(data, window_size):
    """단순 이동 평균 (SMA)"""
    kernel = np.ones(window_size) / window_size
    return np.convolve(data, kernel, mode='same')

# 4시간(240분) 이동 평균 → 일간 변동 제거
MA_WINDOW = 240
trend_ma = moving_average(trend_1x, MA_WINDOW)

# 24시간(1440분) 이동 평균 → 장기 추세만 추출
MA_WINDOW_LONG = 1440
trend_ma_long = moving_average(trend_1x, MA_WINDOW_LONG)

# ── 선형 회귀 (장기 추세) ──
X_reg = trend_days.reshape(-1, 1)
y_reg = trend_ma_long

model = LinearRegression()
model.fit(X_reg, y_reg)

slope = model.coef_[0]
intercept = model.intercept_

print(f"선형 회귀 결과:")
print(f"  - 기울기(slope): {slope:.4f} mm/s per day")
print(f"  - 절편(intercept): {intercept:.4f} mm/s")
print(f"  - 일일 열화량: {slope:.4f} mm/s/day")

# ── 임계값 도달 시점 예측 ──
thresholds = {
    'Zone B (2.5 mm/s)': ISO_ZONE_A,
    'Zone C (6.3 mm/s)': ISO_ZONE_B,
    'Zone D (10.0 mm/s)': ISO_ZONE_C,
}

print(f"\n임계값 도달 예측 (선형 회귀 기반):")
for label, threshold in thresholds.items():
    if slope > 0:
        days_to_threshold = (threshold - intercept) / slope
        if days_to_threshold > 0:
            print(f"  - {label}: Day {days_to_threshold:.1f} (현재 Day {trend_days[-1]:.0f})")
            if days_to_threshold > trend_days[-1]:
                remaining = days_to_threshold - trend_days[-1]
                print(f"    → 잔여 수명: {remaining:.1f}일")
            else:
                print(f"    → 이미 초과함 (Day {trend_days[-1]:.0f} 기준)")
        else:
            print(f"  - {label}: 시작 시점부터 초과")
    else:
        print(f"  - {label}: 열화 추세 없음 (기울기 ≤ 0)")

# ── 지수 회귀 (보다 정확한 예측) ──
# log(y) = a*x + b → y = exp(b) * exp(a*x)
valid_mask = trend_ma_long > 0
log_y = np.log(trend_ma_long[valid_mask])
X_exp = trend_days[valid_mask].reshape(-1, 1)

model_exp = LinearRegression()
model_exp.fit(X_exp, log_y)

exp_rate = model_exp.coef_[0]
exp_intercept = np.exp(model_exp.intercept_)

print(f"\n지수 회귀 결과:")
print(f"  - 열화율: {exp_rate:.5f} /day")
print(f"  - 초기값: {exp_intercept:.4f} mm/s")

# 지수 모델 기반 임계값 도달 예측
for label, threshold in thresholds.items():
    if exp_rate > 0 and exp_intercept > 0:
        days_exp = np.log(threshold / exp_intercept) / exp_rate
        print(f"  - {label}: Day {days_exp:.1f}")

# 예측 곡선 생성 (향후 30일 포함)
future_days = np.linspace(0, TREND_DAYS + 30, 500)
pred_linear = slope * future_days + intercept
pred_exp = exp_intercept * np.exp(exp_rate * future_days)

print("\n분석 완료")

---
## Step 5. 트렌드 분석 및 잔여 수명 예측

### 분석 기법
1. **이동 평균(Moving Average)**: 단기 변동 제거 → 장기 열화 추세 파악
2. **선형 회귀(Linear Regression)**: 열화 추세에 대한 최적 적합선
3. **임계값 도달 시점 예측**: ISO Zone D (10 mm/s) 진입 예상일 추정

In [ ]:
def generate_trend_data(n_samples=TREND_SAMPLES, days=TREND_DAYS):
    """
    90일간의 1X 진동 진폭 트렌드 데이터를 생성합니다.

    Returns
    -------
    time_days : ndarray — 시간 벡터 (일)
    trend_1x : ndarray — 1X 진동 진폭 (mm/s)
    """
    np.random.seed(42)

    time_minutes = np.arange(n_samples)
    time_days = time_minutes / (24 * 60)

    # 1) 기저 열화: 지수적 증가 (1.2 mm/s → ~12 mm/s over 90 days)
    base_amplitude = 1.2  # 초기 1X 진폭 (mm/s)
    degradation_rate = 0.025  # 열화 속도
    base_degradation = base_amplitude * np.exp(degradation_rate * time_days)

    # 2) 일별 변동: 주간(높음) / 야간(낮음) 패턴
    daily_cycle = 0.3 * np.sin(2 * np.pi * time_days - np.pi/2)  # ±0.3 mm/s

    # 3) 주간 변동: 주말에 부하 감소
    weekly_cycle = 0.15 * np.sin(2 * np.pi * time_days / 7)

    # 4) 돌발 이벤트: 무작위 스파이크 (이물질 투입 등)
    spike_events = np.zeros(n_samples)
    n_spikes = 15  # 90일간 15회 돌발 이벤트
    spike_indices = np.random.choice(n_samples, n_spikes, replace=False)
    for idx in spike_indices:
        spike_amp = np.random.uniform(1.0, 4.0)
        spike_width = np.random.randint(30, 180)  # 30~180분 지속
        end_idx = min(idx + spike_width, n_samples)
        decay = np.exp(-np.arange(end_idx - idx) / (spike_width / 3))
        spike_events[idx:end_idx] += spike_amp * decay

    # 5) 측정 노이즈
    noise = 0.15 * np.random.randn(n_samples)

    # 합성
    trend_1x = base_degradation + daily_cycle + weekly_cycle + spike_events + noise
    trend_1x = np.maximum(trend_1x, 0.1)  # 음수 방지

    return time_days, trend_1x


# ── 트렌드 데이터 생성 ──
trend_days, trend_1x = generate_trend_data()

print(f"트렌드 데이터 생성 완료:")
print(f"  - 총 데이터포인트: {len(trend_1x):,}")
print(f"  - 기간: {trend_days[0]:.1f} ~ {trend_days[-1]:.1f}일")
print(f"  - 시작 진폭: {trend_1x[0]:.2f} mm/s")
print(f"  - 종료 진폭: {trend_1x[-1]:.2f} mm/s")
print(f"  - 최소/최대: {trend_1x.min():.2f} / {trend_1x.max():.2f} mm/s")

# ISO Zone 분포
zones = classify_iso10816(trend_1x)
for z in ['A', 'B', 'C', 'D']:
    count = np.sum(zones == z)
    pct = 100 * count / len(zones)
    print(f"  - Zone {z} ({ISO_LABELS[z]:>10s}): {count:>6,} samples ({pct:5.1f}%)")

---
## Step 4. 90일 트렌드 시뮬레이션

실제 파쇄기 운영을 모사하는 **90일간의 1X 진동 진폭 트렌드**를 생성합니다.

- **데이터 규모**: 90일 × 24시간 × 60분 = **129,600 포인트** (60초 간격)
- **열화 모델**: 지수적 증가 + 일별 변동 + 돌발 이벤트
- **시나리오**: 정상 상태에서 시작하여 점진적으로 악화

### 열화 시뮬레이션 구성요소
1. **기저 열화(Base Degradation)**: 지수 함수 기반 점진적 증가
2. **일별 변동(Daily Variation)**: 부하 변동에 따른 일간 패턴
3. **돌발 이벤트(Random Events)**: 이물질 투입 등으로 인한 급격한 변동
4. **측정 노이즈(Measurement Noise)**: 센서 측정 오차

In [ ]:
def classify_iso10816(vibration_rms):
    """
    ISO 10816-3 기준으로 진동 심각도를 분류합니다.

    Parameters
    ----------
    vibration_rms : float or ndarray — 진동 속도 (mm/s RMS)

    Returns
    -------
    str or ndarray — Zone 분류 ('A', 'B', 'C', 'D')
    """
    if np.isscalar(vibration_rms):
        if vibration_rms < ISO_ZONE_A:
            return 'A'
        elif vibration_rms < ISO_ZONE_B:
            return 'B'
        elif vibration_rms < ISO_ZONE_C:
            return 'C'
        else:
            return 'D'
    else:
        arr = np.array(vibration_rms)
        zones = np.full(arr.shape, 'D', dtype='U1')
        zones[arr < ISO_ZONE_C] = 'C'
        zones[arr < ISO_ZONE_B] = 'B'
        zones[arr < ISO_ZONE_A] = 'A'
        return zones


ISO_COLORS = {'A': '#2ecc71', 'B': '#f1c40f', 'C': '#e67e22', 'D': '#e74c3c'}
ISO_LABELS = {'A': 'Good', 'B': 'Acceptable', 'C': 'Warning', 'D': 'Danger'}

# ── 각 시나리오 분류 ──
print(f"{'Scenario':>10s} | {'1X Amp (mm/s)':>14s} | {'ISO Zone':>8s} | {'Status':>12s}")
print("-" * 56)

for name, data in fft_results.items():
    amp_1x = data['harmonics']['1X'][1]
    zone = classify_iso10816(amp_1x)
    print(f"{name:>10s} | {amp_1x:>13.2f} | Zone {zone:>2s} | {ISO_LABELS[zone]:>12s}")

print("\nISO 10816-3 분류 완료")

---
## Step 3. ISO 10816-3 진동 심각도 분류

**ISO 10816-3**은 산업용 회전 기계(15kW 초과)의 진동 심각도를 4개 구간으로 분류하는 국제 표준입니다.

| Zone | 진동 속도 (mm/s RMS) | 상태 | 조치 |
|------|---------------------|------|------|
| **A** (Good) | < 2.5 | 양호 | 정상 운전 |
| **B** (Acceptable) | 2.5 ~ 6.3 | 허용 | 모니터링 강화 |
| **C** (Warning) | 6.3 ~ 10.0 | 경고 | 정비 계획 수립 |
| **D** (Danger) | > 10.0 | 위험 | **즉시 정지 및 정비** |

> **참고**: 파쇄기는 Class III (대형 회전 기계, 유연 지지대) 또는 Class IV에 해당합니다.  
> 본 분석에서는 1X 진폭(mm/s peak)을 기준으로 분류합니다.

In [ ]:
def compute_fft(signal, fs=FS):
    """
    FFT를 계산하고 단측(single-sided) 스펙트럼을 반환합니다.

    Parameters
    ----------
    signal : ndarray — 시간 영역 신호
    fs : int — 샘플링 주파수

    Returns
    -------
    freqs : ndarray — 주파수 벡터 (Hz)
    magnitude : ndarray — 진폭 스펙트럼 (mm/s)
    """
    N = len(signal)
    # Hanning 윈도우 적용
    win = windows.hann(N)
    windowed = signal * win

    # FFT 계산
    yf = fft(windowed)
    freqs = fftfreq(N, 1.0 / fs)

    # 단측 스펙트럼 (양의 주파수만)
    pos_mask = freqs >= 0
    freqs = freqs[pos_mask]
    magnitude = 2.0 / N * np.abs(yf[pos_mask])

    # 윈도우 보정 계수 (Hanning window amplitude correction)
    magnitude *= 2.0  # coherent gain correction for Hanning

    return freqs, magnitude


def extract_harmonic_amplitudes(freqs, magnitude, f1x=F1X_A, search_width=2.0):
    """
    1X, 2X, 3X 주파수 근처의 피크 진폭을 추출합니다.

    Parameters
    ----------
    freqs : ndarray — 주파수 벡터
    magnitude : ndarray — 진폭 스펙트럼
    f1x : float — 기본 회전 주파수 (Hz)
    search_width : float — 피크 탐색 범위 (±Hz)

    Returns
    -------
    dict — {harmonic: (frequency, amplitude)}
    """
    results = {}
    for harmonic, label in [(1, '1X'), (2, '2X'), (3, '3X')]:
        target_f = harmonic * f1x
        mask = (freqs >= target_f - search_width) & (freqs <= target_f + search_width)
        if np.any(mask):
            idx = np.argmax(magnitude[mask])
            freq_val = freqs[mask][idx]
            amp_val = magnitude[mask][idx]
            results[label] = (freq_val, amp_val)
        else:
            results[label] = (target_f, 0.0)
    return results


# ── 전 시나리오 FFT 분석 ──
fft_results = {}
print(f"{'Scenario':>10s} | {'1X Freq':>8s} {'1X Amp':>8s} | {'2X Freq':>8s} {'2X Amp':>8s} | {'3X Freq':>8s} {'3X Amp':>8s}")
print("-" * 82)

for name, data in signals.items():
    freqs, mag = compute_fft(data['signal'])
    harmonics = extract_harmonic_amplitudes(freqs, mag)
    fft_results[name] = {'freqs': freqs, 'magnitude': mag, 'harmonics': harmonics}

    h = harmonics
    print(f"{name:>10s} | "
          f"{h['1X'][0]:7.1f}Hz {h['1X'][1]:7.2f} | "
          f"{h['2X'][0]:7.1f}Hz {h['2X'][1]:7.2f} | "
          f"{h['3X'][0]:7.1f}Hz {h['3X'][1]:7.2f}")

print("\nFFT 분석 완료 — 1X 진폭이 불균형 심각도의 핵심 지표입니다.")

---
## Step 2. FFT 분석 (Fast Fourier Transform)

FFT를 통해 시간 영역 신호를 주파수 영역으로 변환하고, **1X / 2X / 3X** 성분의 진폭을 추출합니다.

- **1X (기본 회전 주파수)**: 축 불균형의 **핵심 지표** — 진폭이 클수록 불균형 심각
- **2X (2배 주파수)**: 미스얼라인먼트(축 정렬 불량) 관련
- **3X (3배 주파수)**: 구조적 공진, 베어링 결함 등

> **Hanning Window**를 적용하여 스펙트럴 누설(spectral leakage)을 최소화합니다.

In [ ]:
# ── 시나리오 정의 ──
SCENARIOS = {
    'Normal':   {'1X': 1.5,  '2X_ratio': 0.30, '3X_ratio': 0.10, 'noise': 0.3,  'color': '#2ecc71'},
    'Slight':   {'1X': 3.5,  '2X_ratio': 0.35, '3X_ratio': 0.12, 'noise': 0.5,  'color': '#f1c40f'},
    'Moderate': {'1X': 6.0,  '2X_ratio': 0.40, '3X_ratio': 0.15, 'noise': 0.8,  'color': '#e67e22'},
    'Severe':   {'1X': 9.0,  '2X_ratio': 0.45, '3X_ratio': 0.18, 'noise': 1.2,  'color': '#e74c3c'},
    'Critical': {'1X': 14.0, '2X_ratio': 0.50, '3X_ratio': 0.20, 'noise': 2.0,  'color': '#8e44ad'},
}

def generate_vibration_signal(scenario_params, f1x=F1X_A, fs=FS, duration=T_WINDOW):
    """
    축 불균형 진동 신호를 합성합니다.

    Parameters
    ----------
    scenario_params : dict  — 1X 진폭, 2X/3X 비율, 노이즈 레벨
    f1x : float             — 1X 회전 주파수 (Hz)
    fs : int                — 샘플링 주파수 (Hz)
    duration : float        — 신호 길이 (초)

    Returns
    -------
    t : ndarray — 시간 벡터
    signal : ndarray — 합성 진동 신호 (mm/s)
    """
    n_samples = int(fs * duration)
    t = np.arange(n_samples) / fs

    amp_1x = scenario_params['1X']
    amp_2x = amp_1x * scenario_params['2X_ratio']
    amp_3x = amp_1x * scenario_params['3X_ratio']
    noise_std = scenario_params['noise']

    # 위상 랜덤화 (실제 신호 모사)
    phi1 = np.random.uniform(0, 2 * np.pi)
    phi2 = np.random.uniform(0, 2 * np.pi)
    phi3 = np.random.uniform(0, 2 * np.pi)

    # 합성 신호: 1X + 2X + 3X + noise
    signal = (
        amp_1x * np.sin(2 * np.pi * f1x * t + phi1) +       # 1X: 축 불균형
        amp_2x * np.sin(2 * np.pi * 2 * f1x * t + phi2) +   # 2X: 미스얼라인먼트
        amp_3x * np.sin(2 * np.pi * 3 * f1x * t + phi3) +   # 3X: 구조 공진
        noise_std * np.random.randn(n_samples)                 # 배경 노이즈
    )

    return t, signal

# ── 전 시나리오 신호 생성 ──
signals = {}
for name, params in SCENARIOS.items():
    t, sig = generate_vibration_signal(params)
    signals[name] = {'t': t, 'signal': sig, 'params': params}
    rms = np.sqrt(np.mean(sig**2))
    peak = np.max(np.abs(sig))
    print(f"[{name:>8s}] 1X={params['1X']:5.1f} mm/s | RMS={rms:.2f} mm/s | Peak={peak:.2f} mm/s")

print(f"\n총 {len(signals)}개 시나리오 × {N_FFT:,} samples = {len(signals)*N_FFT:,} 데이터포인트 생성 완료")

---
## Step 1. 진동 신호 생성 (Vibration Signal Synthesis)

파쇄기 축의 진동 신호를 **5가지 불균형 시나리오**로 합성합니다.

### 불균형 시나리오 정의

| 시나리오 | 상태 | 1X 진폭 (mm/s) | 설명 |
|---------|------|----------------|------|
| Normal | 정상 | 1.5 | 신품 상태 |
| Slight | 경미 | 3.5 | 초기 마모 |
| Moderate | 보통 | 6.0 | 주의 필요 |
| Severe | 심각 | 9.0 | 정비 필요 |
| Critical | 위험 | 14.0 | 즉시 정지 |

### 신호 구성
- **1X 성분**: 축 회전 기본 주파수 (불균형의 주 지표)
- **2X 성분**: 미스얼라인먼트 관련 (1X의 30~50%)
- **3X 성분**: 구조적 공진 (1X의 10~20%)
- **배경 노이즈**: 가우시안 백색 잡음

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.fft import fft, fftfreq
from scipy.signal import welch, windows
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# ── 글로벌 설정 ──
np.random.seed(42)
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 100,
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# ── 상수 정의 ──
FS = 10_000            # 샘플링 주파수 (Hz)
T_WINDOW = 1.0         # FFT 윈도우 길이 (초)
N_FFT = int(FS * T_WINDOW)  # FFT 포인트 수

RPM_A = 1200           # A축 회전수
RPM_B = 800            # B축 회전수
F1X_A = RPM_A / 60     # A축 1X 주파수 = 20 Hz
F1X_B = RPM_B / 60     # B축 1X 주파수 ≈ 13.33 Hz

# ISO 10816-3 경계값 (mm/s RMS)
ISO_ZONE_A = 2.5       # Good
ISO_ZONE_B = 6.3       # Acceptable
ISO_ZONE_C = 10.0      # Warning
# Zone D: > 10.0       # Danger

# 90일 트렌드 파라미터
TREND_DAYS = 90
TREND_INTERVAL_SEC = 60
TREND_SAMPLES = TREND_DAYS * 24 * 60  # 129,600

print(f"샘플링 주파수: {FS:,} Hz")
print(f"FFT 윈도우: {N_FFT:,} samples ({T_WINDOW}s)")
print(f"A축 회전 주파수: {F1X_A:.1f} Hz ({RPM_A} RPM)")
print(f"B축 회전 주파수: {F1X_B:.2f} Hz ({RPM_B} RPM)")
print(f"트렌드 데이터: {TREND_DAYS}일 × {24*60} samples/day = {TREND_SAMPLES:,} samples")
print("설정 완료!")

In [ ]:
!pip install -q scipy scikit-learn

## Step 0. 라이브러리 설치 및 임포트

# 🏭 Classical DSP 축 불균형 진단 — 파쇄기(Shredder)

## 프로젝트 개요

**파쇄기 축 불균형(Shaft Imbalance)**은 회전 기계의 가장 흔한 고장 원인 중 하나입니다.  
본 노트북에서는 **고전적 디지털 신호 처리(Classical DSP)** 기법을 활용하여:

1. **진동 신호 생성** — 10kHz 샘플링, 5단계 불균형 시나리오
2. **FFT 분석** — 1X, 2X, 3X 주파수 성분 추출
3. **ISO 10816-3 기준 분류** — 진동 심각도 4개 구간
4. **90일 트렌드 시뮬레이션** — 129,600개 데이터포인트 (60초 간격)
5. **잔여 수명 예측** — 선형 회귀 기반 임계값 도달 시점 추정

을 **생산 현장 수준(Production-level)**으로 구현합니다.

---

| 항목 | 사양 |
|------|------|
| 샘플링 주파수 | 10,000 Hz |
| FFT 윈도우 | 1초 (10,000 samples) |
| 트렌드 데이터 | 90일 × 1,440 samples/day = **129,600 samples** |
| 기준 규격 | ISO 10816-3 (산업용 회전기계) |
| 파쇄기 축 RPM | A축: 1,200 RPM (20 Hz), B축: 800 RPM (13.3 Hz) |